In [1]:
import logfire

from src.agents.hybrid_search import HybridSearch
from src.common.services.qdrant import QdrantStorageService
from src.common.services.reranker import Reranker
from src.common.utils.config import config
from src.ingestion.chunking.chunker_factory import create_chunker
from src.ingestion.chunking.chunking_config import ChunkingConfig
from src.ingestion.embedding import EmbeddingService

/home/mano/Manoj/Learning/k_academy/advanced_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logfire.configure(service_name="qdrant_test")

Logfire project URL: https://logfire-us.pydantic.dev/manojee/studious

In [3]:
qt = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=config.VECTOR_SIZE,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [4]:
em = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME, dimensions=config.EMBEDDING_DIMENSIONS
)

In [ ]:
content_list = "Python Machine Learning Interview Prep — Part 1\n\nPython, ML Fundamentals, Supervised/Unsupervised, Data Wrangling & Benchmarking\n\nTarget Role:\n\nPython ML Engineer |\n\nExperience:\n\n2–3 Years\n\n1."

In [ ]:
chunking_config = ChunkingConfig(type="recursive_character", size=512, overlap=64)
chunker = create_chunker(chunking_config)
chunks = chunker.chunk(content_list)

In [ ]:
em_chunk = await em.embed_chunks(chunks)

In [ ]:
await qt.ping()

In [ ]:
await qt.upsert_embedded_chunks(embedded_chunks=em_chunk)

In [5]:
ques = [
    "Explain the difference between `deepcopy` and `copy` in the context of ML model parameters?"
]

In [ ]:
em_query = await em.embed_single(ques[0])

In [ ]:
q_search = await qt.search(query=ques[0], query_vector=em_query)

In [ ]:
q_search

In [6]:
hydrid = HybridSearch(storage_service=qt, embedding_service=em)

In [7]:
final = await hydrid.search(queries=ques)
print(len(final))
print(final[0])

13:22:28.822 hybrid_search_start
13:22:28.825 single_search
13:22:32.970 hybrid_search_result
13:22:32.970 hybrid_search_complete
11
{'text': 'Python Machine Learning Interview Prep — Part 1\n\nPython, ML Fundamentals, Supervised/Unsupervised, Data Wrangling & Benchmarking\n\nTarget Role:\n\nPython ML Engineer |\n\nExperience:\n\n2–3 Years\n\n1. Python & ML Libraries\n\nQ1: What are Python generators and how are they useful in ML data pipelines?\n\nAnswer:\n\nGenerators are functions that use\n\nyield\n\nto produce a sequence of values lazily, one at a time, without storing the entire sequence in memory. In ML, they\'re critical for streaming large datasets that don\'t fit in memory.\n\n# Generator for batch-loading a large CSV\nimport pandas as pd\n\ndef data_generator(filepath, chunk_size=1000):\n    for chunk in pd.read_csv(filepath, chunksize=chunk_size):\n        X = chunk.drop("target", axis=1).values\n        y = chunk["target"].values\n        yield X, y\n\n# Usage in training 

In [8]:
rerank = Reranker()

In [10]:
top_result = await rerank.rerank(query=ques[0], candidates=final)

INFO:flashrank.Ranker:Downloading ms-marco-TinyBERT-L-2-v2...


13:23:00.165 reranking_operation
13:23:00.167   initializing_reranker_model
13:23:00.167   ranker_model_loading


ms-marco-TinyBERT-L-2-v2.zip: 100%|██████████| 3.26M/3.26M [00:02<00:00, 1.24MiB/s]


13:23:14.585     reranker_model_loaded_successfully
13:23:14.586   rerank_preparation


INFO:numexpr.utils:NumExpr defaulting to 8 threads.


13:23:15.296   rerank_result_merging
13:23:15.296   rerank_complete


In [11]:
top_result

[{'text': 'Python Machine Learning Interview Prep — Part 1\n\nPython, ML Fundamentals, Supervised/Unsupervised, Data Wrangling & Benchmarking\n\nTarget Role:\n\nPython ML Engineer |\n\nExperience:\n\n2–3 Years\n\n1. Python & ML Libraries\n\nQ1: What are Python generators and how are they useful in ML data pipelines?\n\nAnswer:\n\nGenerators are functions that use\n\nyield\n\nto produce a sequence of values lazily, one at a time, without storing the entire sequence in memory. In ML, they\'re critical for streaming large datasets that don\'t fit in memory.\n\n# Generator for batch-loading a large CSV\nimport pandas as pd\n\ndef data_generator(filepath, chunk_size=1000):\n    for chunk in pd.read_csv(filepath, chunksize=chunk_size):\n        X = chunk.drop("target", axis=1).values\n        y = chunk["target"].values\n        yield X, y\n\n# Usage in training loop\nfor X_batch, y_batch in data_generator("large_data.csv"):\n    model.partial_fit(X_batch, y_batch)\n\nKey benefits:\n\nMemory 